

###### ***BLOCK 1 — TYPE: MARKDOWN***

# ***Preprocesamiento TUSZ v2.0.3 (Binary: bckg vs seizure)***

Este notebook genera segmentos EEG en ventanas de duración fija (por defecto 4 segundos),
aplicando filtrado (bandpass + notch), remuestreo a 250 Hz, lectura de etiquetas desde `.csv_bi`,
y segmentación con solapamiento controlado (bckg sin solapamiento, seizure con solapamiento).

**Objetivo principal de este refactor:**
1. Centralizar el preprocesamiento en `RAIZ/preprocesamiento/` (un solo pipeline compartido).
2. Guardar los `.npy` en `RAIZ/data_procesada/TUSZ_processed_binary_individual_segments/segment_interval_4_sec/`.
3. Seleccionar pacientes (por ahora) con el criterio:
   **"cantidad total de minutos/segundos de seizure"** para mitigar desbalance.


In [ ]:
# BLOCK 2 — TYPE: CODE

import os
import sys

# === RAIZ del repositorio/proyecto ===
# Notebook ubicado en: RAIZ/preprocesamiento/notebook/
# Por tanto, RAIZ está 2 niveles arriba.
RAIZ: str = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
print("RAIZ:", RAIZ)

# === Agregar src de preprocesamiento al path ===
PREPROCESS_SRC: str = os.path.join(RAIZ, "preprocesamiento", "src")
if PREPROCESS_SRC not in sys.path:
    sys.path.insert(0, PREPROCESS_SRC)

print("PREPROCESS_SRC:", PREPROCESS_SRC)

# Importar funciones del módulo refactorizado
from data_reader_2023 import *

RAIZ: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis
PREPROCESS_SRC: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/preprocesamiento/src


In [ ]:
# TYPE: CODE 2.1

%load_ext autoreload
%autoreload 2
# Recarga automáticamente todos los módulos importados.

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# TYPE: CODE 2.2
# Probaremos la funcion cubo que esta en src/data_reader_2023.py
a: int = 2
print(cubo(a))

9



###### ***BLOCK 3 — TYPE: MARKDOWN***
## ***Parámetros globales del preprocesamiento***

- En esta sección se definen los valores por defecto del pipeline (filtros, remuestreo,
segmentación y configuración binaria/multiclase).  
- another


In [24]:
# TYPE: CODE 4

from typing import List, Literal, Tuple
import numpy as np
from scipy.signal import iirnotch

# ====== Constantes (NO CAMBIAR valores) ======
# Define bandpass filter constants
lowcut: float = 0.5
highcut: float = 120.0
fs: int = 1024
resampleFS: int = 250

# Define bandpass filter constants
notch_1_b: np.ndarray
notch_1_a: np.ndarray
notch_1_b, notch_1_a = iirnotch(1.0, Q=30.0, fs=resampleFS)

notch_60_b: np.ndarray
notch_60_a: np.ndarray
notch_60_b, notch_60_a = iirnotch(60.0, Q=30.0, fs=resampleFS)

# Define segment interval length in sec
segment_interval: int = 4
print("Segment Interval:", segment_interval)

binary_classifier_flag: bool = True

if binary_classifier_flag:
    seizure_types: List[str] = ["bckg", "seizure"]
    seizure_session_downsampling_ratio: List[float] = [1.0, 1.0]
    seizure_overlapping_ratio: List[float] = [0.0, 0.75]
else:
    seizure_types: List[str] = ["fnsz", "gnsz", "cpsz", "bckg"]
    seizure_session_downsampling_ratio: List[float] = [1.0, 1.0, 1.0, 1.0]
    seizure_overlapping_ratio: List[float] = [0.75, 0.75, 0.75, 0.0]

# Modos de datos permitidos
DataMode = Literal["tiny", "small", "large", "full"]
data_mode: DataMode = "tiny"

Segment Interval: 4


###### ***BLOCK 5 — TYPE: MARKDOWN***
## ***Rutas de entrada y salida (refactor)***

- **Entrada (EDF):** `RAIZ/dataset/tuh_eeg_seizure/v2.0.3/edf/{train,dev,eval}`
- **Salida (.npy):** `RAIZ/data_procesada/TUSZ_processed_binary_individual_segments/segment_interval_4_sec/`
  con estructura:
  - train/bckg, train/seizure
  - val/bckg, val/seizure
  - test/bckg, test/seizure


In [ ]:
# TYPE: CODE 6

import os
import shutil

# === Entrada EDF ===
train_val_root: str = os.path.join(
    RAIZ,
    "dataset",
    "tuh_eeg_seizure",
    "v2.0.3",
    "edf",
    "train"
)

dev_root: str = os.path.join(
    RAIZ,
    "dataset",
    "tuh_eeg_seizure",
    "v2.0.3",
    "edf",
    "dev"
)

eval_root: str = os.path.join(
    RAIZ,
    "dataset",
    "tuh_eeg_seizure",
    "v2.0.3",
    "edf",
    "eval"
)

print("Train root:", train_val_root)
print("Dev root:", dev_root)
print("Eval root:", eval_root)

# === Salida procesada (.npy) ===
print("Binary classifier flag:", binary_classifier_flag)
if binary_classifier_flag:
    save_root: str = os.path.join(
        RAIZ,
        "data_procesada",
        "TUSZ_processed_binary_individual_segments"
    )
else:
    save_root: str = os.path.join(
        RAIZ,
        "data_procesada",
        "TUSZ_processed_multiclass_individual_segments"
    )

print("Save root:", save_root)
if not os.path.exists(save_root):
    os.makedirs(save_root, exist_ok=True)
    print(f"Created {save_root} directory.")
else:
    print(f"Ojo, {save_root} directory already exists.")

Train root: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/dataset/tuh_eeg_seizure/v2.0.3/edf/train
Dev root: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/dataset/tuh_eeg_seizure/v2.0.3/edf/dev
Eval root: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/dataset/tuh_eeg_seizure/v2.0.3/edf/eval
Binary classifier flag: True
Save root: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/data_procesada/TUSZ_processed_binary_individual_segments
Ojo, /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/data_procesada/TUSZ_processed_binary_individual_segments directory already exists.


In [35]:
# TYPE: CODE 6.1
# To delete previous data repo when running a new experiment
import os
import shutil

# Construye el path del directorio para este experimento
segment_folder: str = os.path.join(
    save_root,
    f"segment_interval_{segment_interval}_sec"
)
print("Segment folder:", segment_folder)

# Si no existe, lo creamos (experimento nuevo)
if not os.path.exists(segment_folder):
    print("Creating new segment folder:", segment_folder)
    os.makedirs(segment_folder, exist_ok=True)
else:
    # Si ya existe, borramos todo su contenido para evitar duplicados
    # al concatenar archivos .npy en ejecuciones sucesivas
    print("Deleting existing segment folder:", segment_folder)
    filenames: List[str] = os.listdir(segment_folder)

    for filename in filenames:
        print("filename to delete:", filename)
        file_path:str  = os.path.join(segment_folder, filename)
        try:
            # Si es archivo o enlace simbólico, lo eliminamos
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            # Si es un directorio, lo borramos recursivamente
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            # Captura cualquier error en la eliminación y lo informa
            error_msg: str = f"Failed to delete {file_path}. Reason: {e}"
            print(error_msg)

Segment folder: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/data_procesada/TUSZ_processed_binary_individual_segments/segment_interval_4_sec
Deleting existing segment folder: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/data_procesada/TUSZ_processed_binary_individual_segments/segment_interval_4_sec


###### ***BLOCK 7 — TYPE: MARKDOWN***
## ***Indexación del dataset (listar EDFs y pacientes)***

En esta sección recorremos las carpetas `edf/train`, `edf/dev` y `edf/eval` para:

1. Obtener la lista de paths `.edf`.
2. Obtener la lista de pacientes.
3. Contar tipos de referencia (para inspección).


In [ ]:
from typing import Dict, List, Tuple
import time

print("Indexando sesiones EDF...")

# Obtener rutas de sesión, lista de pacientes y conteo de tipos de referencia
# Train/Val: vienen de edf/train

train_val_paths: List[str]
train_val_patients: List[str]
train_val_reference_type_count: Dict[str, int]
print("Getting all TUSZ 2023 session paths of train_val_root...")

t0:float = time.time()
train_val_paths, train_val_patients, train_val_reference_type_count = get_all_TUSZ_2023_session_paths(train_val_root)
t1 = time.time()

print(f"[train] tiempo indexado: {t1 - t0:.2f} seconds")
print(f"[train] total edfs: {len(train_val_paths)}")
print(f"[train] total pacientes: {len(train_val_patients)}")
print(f"[train] referencias: {train_val_reference_type_count}")


# Test: por ahora usaremos dev como test (igual que tu notebook anterior)

t0 = time.time()
dev_paths, dev_patients, dev_reference_type_count = get_all_TUSZ_2023_session_paths(
    dev_root
)
t1 = time.time()

print(f"[dev] tiempo indexado: {t1 - t0:.2f} s")
print(f"[dev] total edfs: {len(dev_paths)}")
print(f"[dev] total pacientes: {len(dev_patients)}")
print(f"[dev] referencias: {dev_reference_type_count}")

# Eval (opcional): si no existe o está vacío, no pasa nada; lo dejamos preparado
eval_paths: List[str] = []
eval_patients: List[str] = []
eval_reference_type_count: Dict[str, int] = {}

if os.path.exists(eval_root):
    t0 = time.time()
    eval_paths, eval_patients, eval_reference_type_count = (
        get_all_TUSZ_2023_session_paths(eval_root)
    )
    t1 = time.time()
    print(f"[eval] tiempo indexado: {t1 - t0:.2f} s")
    print(f"[eval] total edfs: {len(eval_paths)}")
    print(f"[eval] total pacientes: {len(eval_patients)}")
    print(f"[eval] referencias: {eval_reference_type_count}")
else:
    print("[eval] carpeta eval_root no existe (OK si aún no la tienes).")

Indexando sesiones EDF...
Getting all TUSZ 2023 session paths of train_val_root...
[train] tiempo indexado: 0.05 seconds
[train] total edfs: 4667
[train] total pacientes: 579
[train] referencias: {'01_tcp_ar': 683, '02_tcp_le': 324, '03_tcp_ar_a': 168}


SyntaxError: 'return' outside function (2599350873.py, line 23)

pip install --index-url https://download.pytorch.org/whl/cu118 \
  torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1